## Test the new FolpsD model with bug fixed

In [1]:
import os
import sys
import itertools
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from desilike.samples import MCSamples as Chain

# This notebook only loads/plots saved results; keep JAX from probing or reserving GPU memory.
os.environ['JAX_PLATFORMS'] = 'cpu'
os.environ['JAX_PLATFORM_NAME'] = 'cpu'
os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['MPICH_GPU_SUPPORT_ENABLED'] = '0'

sys.path.append('../..')
import numpy as np
from getdist import MCSamples, plots

import matplotlib as mpl
mpl.rcParams.update({
    "text.usetex": False,
    "mathtext.fontset": "cm",   # Computer Modern-like math
    "font.family": "serif",
})

main_dir = Path('/global/homes/s/shengyu/Y3/desi_y3_redshift_errors/main')
if str(main_dir) not in sys.path:
    sys.path.insert(0, str(main_dir))

from helper import REDSHIFT_BIN_TRACER
from cat_tools import parse_zerr_name
from fit_tools import LikelihoodBuilder, get_fits_fn
from plotting_tools import PLANCK_COSMOLOGY, make_key, chain_fit_results, PARAM_LABELS
from plotting_tools import read_chain

OVERLEAF_DIR = '/global/homes/s/shengyu/Y3/desi_y3_redshift_errors/overleaf/figures'
FITS_DIR = Path('/global/cfs/cdirs/desi/users/shengyu/Y3/full-shape/redshift_errors/fits')

%load_ext autoreload
%autoreload 2

analysis_settings = {
    'smooth_scale_1D': 0.6,
    'smooth_scale_2D': 0.6,
    'fine_bins': 1024,
    'fine_bins_2D': 512,
}

In [2]:
LRG2_set1 = ['/global/cfs/cdirs/desi/users/shengyu/Y3/full-shape/redshift_errors/fix_folpsD_bugs_07_10/fits/cosmo-base_LRG2-z0.6-0.8-mean25-S2+LRG2-z0.6-0.8-mean25-S3-0eee82b3/chain_0.npy', 
             r'$P+B$, Baseline ($B_{202}, \ k_{\rm{max}} = 0.08)$']
# LRG2_new_set1 = ['/global/cfs/cdirs/desi/users/shengyu/Y3/full-shape/redshift_errors/fits/cosmo-base_LRG3-z0.8-1.1-mean25-S2+LRG3-z0.8-1.1-mean25-S3-19754d69/samples/samples_0.h5', 
#                  r'$P+B$, New Baseline ($B_{202}, \ k_{\rm{max}} = 0.08)$']
LRG2_new_set1 = [f'/global/cfs/cdirs/desi/users/shengyu/Y3/full-shape/redshift_errors/fits/cosmo-base_LRG2-z0.6-0.8-mean25-S2+LRG2-z0.6-0.8-mean25-S3-c088f812/samples/samples_0.h5',
                 r'$P+B$, SY damping=tree-level ($B_{202}, \ k_{\rm{max}} = 0.08)$']

LRG2_RY_set1 = [['/global/cfs/cdirs/desi/users/ruiyang/share/test_folps/cosmo-base_LRG2-S2+LRG2-S3-3d3611e2/samples_1.h5',
                 '/global/cfs/cdirs/desi/users/ruiyang/share/test_folps/cosmo-base_LRG2-S2+LRG2-S3-3d3611e2/samples_2.h5',
                 '/global/cfs/cdirs/desi/users/ruiyang/share/test_folps/cosmo-base_LRG2-S2+LRG2-S3-3d3611e2/samples_3.h5',
                 '/global/cfs/cdirs/desi/users/ruiyang/share/test_folps/cosmo-base_LRG2-S2+LRG2-S3-3d3611e2/samples_4.h5'],
                r'$P+B$, RY damping=tree-level ($B_{202}, \ k_{\rm{max}} = 0.08)$'] 

LRG2_RY_set2 = ['/global/cfs/cdirs/desi/users/ruiyang/share/test_folps/cosmo-base_LRG2-S2+LRG2-S3-86a0e1dc/samples_1.h5',
                r'$P+B$, RY damping=None (1-loop-level) ($B_{202}, \ k_{\rm{max}} = 0.08)$'] 


In [3]:
chains = []
labels = []
# params = ['H0', 'Omega_m', 'sigma8_m', 'LRG2.b1', 'LRG2.b2',  'LRG2.b3',  'LRG2.bs', 'LRG2.X_FoG']
params = ['H0', 'Omega_m', 'sigma8_m', 'omega_cdm', 'logA']
# params += ['QSO1.b1p', 'QSO1.b2p', 'QSO1.bsp']
# params += ['QSO1.X_FoG_pp', 'QSO1.alpha0p', 'QSO1.alpha2p', 'QSO1.sn0p', 'QSO1.sn2p']

param_aliases = {
    'LRG2.b2': ['LRG2.b2', 'LRG2.b2p'],
    'LRG2.bs': ['LRG2.bs', 'LRG2.bsp'],
    'LRG2.X_FoG': ['LRG2.X_FoG', 'LRG2.X_FoG_pp'],
}

def get_chain_param_names(chain):
    if hasattr(chain, 'params'):
        return {str(param) for param in chain.params()}
    if hasattr(chain, 'names'):
        return {str(name) for name in chain.names()}
    if hasattr(chain, 'getParamNames'):
        return {str(param.name) for param in chain.getParamNames().names}
    raise AttributeError(f'Cannot find parameter names on {type(chain).__name__}')

def chain_to_getdist_with_aliases(chain, plot_params, aliases=None, settings=None):
    aliases = aliases or {}
    available = get_chain_param_names(chain)
    selected_params = []
    renames = {}
    for target in plot_params:
        candidates = aliases.get(target, [target])
        source = next((candidate for candidate in candidates if candidate in available), None)
        if source is None:
            print(f'Skip missing parameter {target}: tried {candidates}')
            continue
        selected_params.append(source)
        if source != target:
            renames[target] = [source]
    if hasattr(chain, 'to_getdist'):
        gd_chain = chain.to_getdist(params=selected_params, settings=settings)
    else:
        gd_chain = chain.copy()
    if renames:
        gd_chain.updateRenames(renames)
    return gd_chain

for chain_fn, label in [LRG2_new_set1, LRG2_RY_set1, LRG2_RY_set2]:
    chain = read_chain(chain_fn)
    chain = chain_to_getdist_with_aliases(chain, params, aliases=param_aliases, settings=analysis_settings)
    chains.append(chain)
    labels.append(label)

plot_params = params

g = plots.get_subplot_plotter()
g.settings.fig_width_inch = len(plot_params) * 2
g.settings.legend_fontsize = 12
g.settings.axes_labelsize = 20
g.settings.axes_fontsize = 15
g.settings.figure_legend_frame = False
g.settings.num_plot_contours = 2

g.triangle_plot(
    chains,
    plot_params,
    legend_loc='upper right',
    legend_labels=labels,
    contour_colors=['C2', 'C1', 'C0', 'C3', 'C4'],
    contour_lws=1.5,
    contour_ls = ['-', '-', '-', '-.', '-.'],
    filled = [True, False, False, False, False]
)

# from plotting_tools import PLANCK_COSMOLOGY
true_values = PLANCK_COSMOLOGY
for i, pi in enumerate(plot_params[:5]):
    for j, pj in enumerate(plot_params[:i+1]):
        ax = g.subplots[i, j]
        ax.axvline(true_values[pj], c='k', ls=':', lw=1.5)
        if i != j:
            ax.axhline(true_values[pi], c='k', ls=':', lw=1.5)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/global/cfs/cdirs/desi/users/shengyu/Y3/full-shape/redshift_errors/fits/cosmo-base_LRG2-z0.6-0.8-mean25-S2+LRG2-z0.6-0.8-mean25-S3-c088f812/samples/samples_0.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

(24960, 3)